In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Create Widgets ---
dbutils.widgets.removeAll()
dbutils.widgets.text("scale", "1000", "Scale Factor (GB)")
dbutils.widgets.text("warehouse_size", "M", "Warehouse Size (XS, S, M, L, XL, ...)")
dbutils.widgets.text("dbx_pat_token", "", "Databricks PAT Token")
dbutils.widgets.text("dbx_optimize_strategy", "zorder", "Optimize Strategy")
dbutils.widgets.text("dbx_optimize_strategy_slug", "", "Optimize Strategy Slug (auto-computed if blank)")
dbutils.widgets.text("max_clusters", "1", "Max clusters (max_num_clusters for the SQL warehouse)")
dbutils.widgets.text("dbx_hourly_rate", "2.00", "Hourly Rate")
dbutils.widgets.text("benchmark_run_id", "", "Benchmark run ID (auto-populated by orchestrator)")
dbutils.widgets.text("mlflow_experiment_name", "/TPC-DS Benchmark", "MLflow Experiment")

In [ ]:
import os

params = dbutils.widgets.getAll()
benchmark_run_id = params.get("benchmark_run_id", "").strip()
current_path = os.getcwd()
volume_base = f"/Volumes/tpcds_sf_catalog/tpcds/tpcds_raw_data"  # default, overridden by setup cell
warehouse_size = params.get("warehouse_size", "M").strip().upper()
dbx_strategy_raw = params.get("dbx_optimize_strategy", "zorder")
dbx_strategy_slug = params.get("dbx_optimize_strategy_slug", "").strip() or strategy_slug(dbx_strategy_raw)
prefix = f"{warehouse_size}_{params['scale']}_{dbx_strategy_slug}"
result_name_prefix = f"{warehouse_size}_{params['scale']}_{params.get('max_clusters', '1')}_{dbx_strategy_slug}"

# Find or create Databricks SQL warehouse of the right size
host = spark.conf.get("spark.databricks.workspaceUrl")
wh_id, dbx_jdbc_url, warehouse_config = find_or_create_databricks_warehouse(warehouse_size, params["scale"], dbx_strategy_slug, host, params["dbx_pat_token"], max_clusters=int(params.get("max_clusters", "1")))

# MLflow parameters
mlflow_params = {
    "scale": params["scale"],
    "warehouse_size": warehouse_config.get("warehouse_size", warehouse_size),
    "warehouse_name": warehouse_config.get("warehouse_name", ""),
    "warehouse_type": warehouse_config.get("warehouse_type", ""),
    "channel": warehouse_config.get("channel", ""),
    "num_clusters": warehouse_config.get("num_clusters", ""),
    "hourly_rate": params["dbx_hourly_rate"],
    "optimize_strategy": params["dbx_optimize_strategy"],
    "prefix": prefix,
}

In [ ]:
import os, shutil

# Volume base path for all file I/O
volume_base = f"/Volumes/{params.get('dbx_catalog', 'tpcds_sf_catalog')}/{params.get('dbx_schema', 'tpcds')}/{params.get('dbx_volume', 'tpcds_zorder_raw_data')}"
conf_subdir = f"test_files/{prefix}" if prefix else "test_files"
src_conf = os.path.join(volume_base, conf_subdir)
tpcds_dir = "/tmp/tpcds"

# Prepare local working directory
if os.path.exists(tpcds_dir):
    shutil.rmtree(tpcds_dir)
os.makedirs(f"{tpcds_dir}/results", exist_ok=True)
os.makedirs(f"{tpcds_dir}/logs", exist_ok=True)
os.makedirs(f"{tpcds_dir}/test_conf", exist_ok=True)

# Copy config files from volume to local /tmp
for f in os.listdir(src_conf):
    src = os.path.join(src_conf, f)
    if os.path.isfile(src):
        shutil.copy2(src, f"{tpcds_dir}/test_conf/")
    elif os.path.isdir(src):
        shutil.copytree(src, os.path.join(tpcds_dir, f), dirs_exist_ok=True)

# Also copy RSA key if it exists (Snowflake auth)
rsa_key = os.path.join(volume_base, "test_files", "rsa_key.p8")
if os.path.exists(rsa_key):
    shutil.copy2(rsa_key, f"{tpcds_dir}/test_conf/")

# Rewrite paths in JMX files to use local directory
for f in os.listdir(f"{tpcds_dir}/test_conf"):
    if f.endswith(".jmx"):
        jmx_path = os.path.join(tpcds_dir, "test_conf", f)
        with open(jmx_path) as fh:
            content = fh.read()
        content = content.replace(src_conf, tpcds_dir)
        content = content.replace(volume_base, tpcds_dir)
        with open(jmx_path, "w") as fh:
            fh.write(content)

print(f"Prepared /tmp/tpcds from {src_conf}")

In [ ]:
import glob

concurrency_streams = sorted(
    glob.glob("/tmp/tpcds/tpcds_databricks_concurrency_streams/stream_*.tsv")
)

run_jmeter_with_tracking(
    platform="databricks",
    test_type="concurrency",
    jmx_file="/tmp/tpcds/test_conf/databricks_concurrency.jmx",
    tsv_files=concurrency_streams,
    jdbc_driver="/tpcds/db_driver.jar",
    result_csvs=[
        "/tmp/tpcds/results/databricks-concurrency-report.csv",
    ],
    log_file="/tmp/tpcds/logs/jmeter-dbx-concurrency.log",
    run_params=mlflow_params,
    experiment_name=params["mlflow_experiment_name"],
    workspace_results_dir=f"{volume_base}/results",
    workspace_logs_dir=f"{volume_base}/logs",
    result_prefix=f"{result_name_prefix}-" if result_name_prefix else "",
    run_id=benchmark_run_id,
)

In [ ]:
# --- Validate Results ---
import csv as _csv_check, os as _os_check

result_prefix = f"{result_name_prefix}-" if result_name_prefix else ""
run_id_suffix = f"__{benchmark_run_id}" if benchmark_run_id else ""
failures = []
result_files_checked = []

for result_file in ["databricks-concurrency-report.csv"]:
    full_name = f"{result_prefix}{result_file.replace('-report.csv', f'{run_id_suffix}-report.csv')}"
    result_files_checked.append(result_file)
    ws_path = _os_check.path.join(volume_base, "results", full_name)
    
    if not _os_check.path.exists(ws_path):
        failures.append(f"{full_name}: file not found")
        continue
    
    with open(ws_path) as f:
        reader = _csv_check.DictReader(f)
        rows = list(reader)
    
    if not rows:
        failures.append(f"{full_name}: empty (0 rows)")
        continue
    
    ok = sum(1 for r in rows if r.get("success") == "true")
    fail = sum(1 for r in rows if r.get("success") == "false")
    
    if fail > 0:
        first_err = next(r for r in rows if r.get("success") == "false")
        failures.append(f"{full_name}: {fail}/{len(rows)} failed (first: {first_err.get('label','?')}: {first_err.get('responseMessage','')[:100]})")
    else:
        print(f"{full_name}: {ok}/{len(rows)} OK")

if failures:
    # Delete partial result files so 6 - Results Comparison's Autoloader does not ingest them.
    for result_file in [r for r in [
            f"{result_prefix}{rf.replace('-report.csv', f'{run_id_suffix}-report.csv')}"
            for rf in result_files_checked
        ] if r]:
        ws_path = _os_check.path.join(volume_base, "results", result_file)
        if _os_check.path.exists(ws_path):
            _os_check.remove(ws_path)
            print(f"deleted partial result: {ws_path}")
    for f in failures:
        print(f"FAIL: {f}")
    raise RuntimeError("FAILED: " + "; ".join(failures))
else:
    print("All results validated successfully")